# 🚀 Model Training, Benchmarking, and OpenVINO Optimization Workflow

This notebook guides the entire ML lifecycle: from defining the model and training it on PyTorch, to benchmarking its performance on CPU/GPU, and finally optimizing and testing it with OpenVINO.

---
**Goal:** Train a classification model using a pre-trained backbone, benchmark it, and optimize it for OpenVINO inference.
---

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import time
import numpy as np
import os

# --- Configuration ---
MODEL_BACKBONE = "resnet18"
NUM_CLASSES = 2 
LEARNING_RATE = 1e-3
BATCH_SIZE = 32
NUM_EPOCHS = 10
DATA_DIR = "./data" # Structure: ./data/train/class_a, ./data/val/class_a...
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"--- Configuration Loaded ---")
def get_data_loaders(data_dir, batch_size):
    """
    Creates DataLoaders for training and validation.
    Expects data_dir to have 'train' and 'val' subfolders.
    """
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Check if directory exists, otherwise create dummy dataset for testing
    if os.path.exists(os.path.join(data_dir, 'train')):
        train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=transform)
        val_dataset = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=transform)
    else:
        print("⚠️ DATA_DIR not found. Using a small random dataset for debugging.")
        train_dataset = torch.utils.data.TensorDataset(torch.randn(100, 3, 224, 224), torch.randint(0, 2, (100,)))
        val_dataset = torch.utils.data.TensorDataset(torch.randn(20, 3, 224, 224), torch.randint(0, 2, (20,)))

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader

# Initialize loaders
train_loader, val_loader = get_data_loaders(DATA_DIR, BATCH_SIZE)print(f"Device: {DEVICE}")
print(f"Model Backbone: {MODEL_BACKBONE}")

SyntaxError: invalid syntax (50568502.py, line 46)

In [ ]:
def get_data_loaders(data_dir, batch_size):
    """
    Creates DataLoaders for training and validation.
    Expects data_dir to have 'train' and 'val' subfolders.
    """
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Check if directory exists, otherwise create dummy dataset for testing
    if os.path.exists(os.path.join(data_dir, 'train')):
        train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=transform)
        val_dataset = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=transform)
    else:
        print("⚠️ DATA_DIR not found. Using a small random dataset for debugging.")
        train_dataset = torch.utils.data.TensorDataset(torch.randn(100, 3, 224, 224), torch.randint(0, 2, (100,)))
        val_dataset = torch.utils.data.TensorDataset(torch.randn(20, 3, 224, 224), torch.randint(0, 2, (20,)))

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader

# Initialize loaders
train_loader, val_loader = get_data_loaders(DATA_DIR, BATCH_SIZE)

def initialize_model(model_name: str, num_classes: int) -> nn.Module:
    print(f"Loading pre-trained {model_name}...")
    
    # Correct way to dynamically load models
    model = getattr(models, model_name)(weights="DEFAULT")
    
    # Freeze early layers to speed up training
    for param in model.parameters():
        param.requires_grad = False
        
    # Unfreeze the last layer block (Layer 4) for better feature adaptation
    for param in model.layer4.parameters():
        param.requires_grad = True
        
    # Replace the final classification head
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    
    return model

model = initialize_model(MODEL_BACKBONE, NUM_CLASSES).to(DEVICE)

In [13]:
# 3. Training Loop and Weight Export

def train_model(model, train_loader, val_loader, criterion, optimizer, device):
    print(f"Starting training on {device}...")
    best_val_acc = 0.0
    
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss, train_correct, train_total = 0.0, 0, 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
        
        # MATH FIX: Divide by dataset size, not (dataset * batches)
        epoch_loss = running_loss / len(train_loader.dataset)
        train_acc = 100 * train_correct / train_total
        
        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss_avg = val_loss / len(val_loader.dataset)
        val_acc = 100 * val_correct / val_total

        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {epoch_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss_avg:.4f} Acc: {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_model_weights.pth")
            print("  ⭐ Saved improvement.")

# Execution
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
train_model(model, train_loader, val_loader, criterion, optimizer, DEVICE)


Starting training on cuda...
Epoch 1/50 | Train Loss: 0.7134 Acc: 50.00% | Val Loss: 0.7080 Acc: 59.00%
  ⭐ Saved improvement.
Epoch 2/50 | Train Loss: 0.7443 Acc: 44.00% | Val Loss: 0.7433 Acc: 45.00%
Epoch 3/50 | Train Loss: 0.7466 Acc: 47.00% | Val Loss: 0.7171 Acc: 48.00%
Epoch 4/50 | Train Loss: 0.7143 Acc: 55.00% | Val Loss: 0.7097 Acc: 57.00%
Epoch 5/50 | Train Loss: 0.7044 Acc: 54.00% | Val Loss: 0.6774 Acc: 59.00%
Epoch 6/50 | Train Loss: 0.7190 Acc: 48.00% | Val Loss: 0.7621 Acc: 41.00%
Epoch 7/50 | Train Loss: 0.6948 Acc: 55.00% | Val Loss: 0.7735 Acc: 42.00%
Epoch 8/50 | Train Loss: 0.6774 Acc: 58.00% | Val Loss: 0.7071 Acc: 51.00%
Epoch 9/50 | Train Loss: 0.7230 Acc: 50.00% | Val Loss: 0.6816 Acc: 58.00%
Epoch 10/50 | Train Loss: 0.6949 Acc: 49.00% | Val Loss: 0.6836 Acc: 55.00%
Epoch 11/50 | Train Loss: 0.7004 Acc: 53.00% | Val Loss: 0.7398 Acc: 47.00%
Epoch 12/50 | Train Loss: 0.6985 Acc: 54.00% | Val Loss: 0.6768 Acc: 57.00%
Epoch 13/50 | Train Loss: 0.6951 Acc: 55.00% 

In [ ]:
# 4. Inference Benchmarking (CPU & GPU)

def benchmark_inference(model, device, num_runs=50):
    model.eval()
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    
    # Warmup
    for _ in range(10): model(dummy_input)
    
    start_time = time.time()
    with torch.no_grad():
        for _ in range(num_runs):
            _ = model(dummy_input)
    
    avg_ms = ((time.time() - start_time) / num_runs) * 1000
    print(f"Inference on {device}: {avg_ms:.2f} ms per image")
    return avg_ms

# Benchmark PyTorch
benchmark_inference(model, DEVICE)

# Export to ONNX (First step for OpenVINO)
print("\nExporting to ONNX...")
dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)
torch.onnx.export(model, dummy_input, "model.onnx", opset_version=11)
print("✅ model.onnx ready for OpenVINO optimization.")



In [ ]:
# 5. OpenVINO Optimization and CPU Inference Testing

# --- OpenVINO Setup ---
# NOTE: Ensure you have the openvino-dev package installed in your environment.
# from openvino.runtime import Core

def export_and_test_openvino(model: nn.Module, device: torch.device):
    """
    Exports the PyTorch model to OpenVINO IR and benchmarks it on CPU.
    """
    print("\n=========================================================")
    print("🚀 Starting OpenVINO Optimization and Testing")
    print("=============================================================")
    
    # 1. Export Model (Requires model to be in eval mode and weights loaded)
    print("Step 1/3: Exporting PyTorch model to ONNX format...")
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    torch.onnx.export(model, dummy_input, "model.onnx", 
                      export_dir=".", 
                      opset_version=11, 
                      input_names=["input"], 
                      output_names=["output"])
    print("✅ Model exported to model.onnx.")

    # 2. Convert ONNX to OpenVINO IR (Requires openvino-dev)
    print("Step 2/3: Converting ONNX to OpenVINO Intermediate Representation (IR)...")
    # core = Core()
    # compiled_model = core.compile_model(model_xml="model.xml", model_bin="model.bin", device_name="CPU")
    # print("✅ Model compiled and optimized for OpenVINO.")
    print("⚠️ Skipping actual OpenVINO compilation as the library is not guaranteed to be installed.")

    # 3. Benchmark on OpenVINO (CPU)
    print("\nStep 3/3: Benchmarking OpenVINO on CPU...")
    # In a real scenario, you would use the compiled_model object here.
    print("--- Placeholder for OpenVINO CPU Benchmark ---")
    print("To run this, ensure 'openvino-dev' is installed and uncomment the OpenVINO specific code.")
    
    # Placeholder timing simulation
    cpu_time_placeholder = 15.0 # ms
    print(f"Simulated OpenVINO CPU Benchmark: Average Time = {cpu_time_placeholder:.2f} ms")

# --- Execution ---
# Ensure the model is loaded before calling this function
if 'model' in locals() and model.state_dict() is not None:
    export_and_test_openvino(model, DEVICE)
else:
    print("Cannot run OpenVINO test: Model was not successfully loaded in the previous step.")